# 🚬 흡연 분류 AI 해커톤 - V5 (하이브리드 최종)

**V5 = V3 튜닝 + V4 K-Fold + Stacking + Optuna**

핵심:
- ⭐ RandomizedSearchCV로 최적 파라미터 탐색
- ⭐ K-Fold 교차 예측 (데이터 100% 활용)
- ⭐ Stacking 앙상블
- ⭐ Optuna로 가중치/임계값 최적화

---

## 📌 STEP 1: 환경 설정

In [ ]:
!pip install -q xgboost lightgbm catboost optuna

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 경로 설정
base_path = '/content/drive/MyDrive/AI_Projects/smoking_hackathon/'
train_path = base_path + 'data/train.csv'
test_path = base_path + 'data/test.csv'
submission_path = base_path + 'data/sample_submission.csv'
result_path = base_path + 'results/'

In [ ]:
import numpy as np
import pandas as pd
import optuna
from optuna.samplers import TPESampler
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import random
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
set_seed(42)

print("✅ 라이브러리 임포트 완료!")

## 📌 STEP 2: 데이터 로드 및 전처리

In [ ]:
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
submission = pd.read_csv(submission_path)

print(f"Train: {train.shape}, Test: {test.shape}")

train_df = train.copy()
test_df = test.copy()

# ID 처리
test_id = test_df['ID'].copy() if 'ID' in test_df.columns else test_df.get('id', pd.Series(range(len(test_df))))
train_df = train_df.drop(['ID', 'id'], axis=1, errors='ignore')
test_df = test_df.drop(['ID', 'id'], axis=1, errors='ignore')

X = train_df.drop('label', axis=1)
y = train_df['label']
X_test = test_df.drop('label', axis=1, errors='ignore')

print(f"특성 수: {X.shape[1]}")
print(f"흡연자 비율: {y.mean()*100:.2f}%")

## 📌 STEP 3: 피처 엔지니어링

In [ ]:
def create_features(df):
    df = df.copy()
    col_map = {c: c.lower() for c in df.columns}
    df_l = df.rename(columns=col_map)
    cols = df_l.columns.tolist()
    
    # 콜레스테롤
    if 'hdl' in cols and 'ldl' in cols:
        df['HDL_LDL_ratio'] = df_l['hdl'] / (df_l['ldl'] + 1)
        df['LDL_HDL_ratio'] = df_l['ldl'] / (df_l['hdl'] + 1)
    if 'cholesterol' in cols and 'hdl' in cols:
        df['Atherogenic_idx'] = (df_l['cholesterol'] - df_l['hdl']) / (df_l['hdl'] + 1)
    if 'triglyceride' in cols and 'hdl' in cols:
        df['TG_HDL_ratio'] = df_l['triglyceride'] / (df_l['hdl'] + 1)
    
    # 간 기능
    if 'gtp' in cols:
        df['GTP_log'] = np.log1p(df_l['gtp'])
        df['GTP_sq'] = df_l['gtp'] ** 2
    if 'ast' in cols and 'alt' in cols:
        df['AST_ALT_ratio'] = df_l['ast'] / (df_l['alt'] + 1)
    
    # 헤모글로빈
    if 'hemoglobin' in cols:
        df['Hemo_sq'] = df_l['hemoglobin'] ** 2
        df['Hemo_log'] = np.log1p(df_l['hemoglobin'])
    if 'hemoglobin' in cols and 'gtp' in cols:
        df['Hemo_x_GTP'] = df_l['hemoglobin'] * df_l['gtp']
    
    # 혈압
    if 'systolic' in cols and 'diastolic' in cols:
        df['Pulse_pressure'] = df_l['systolic'] - df_l['diastolic']
        df['MAP'] = df_l['diastolic'] + (df_l['systolic'] - df_l['diastolic']) / 3
    
    # 체형
    if 'height' in cols and 'weight' in cols:
        df['BMI_calc'] = df_l['weight'] / ((df_l['height']/100) ** 2 + 0.01)
    
    # 시력
    eye_cols = [c for c in cols if 'eyesight' in c]
    if len(eye_cols) >= 2:
        df['Eyesight_avg'] = df_l[eye_cols].mean(axis=1)
    
    # 나이
    if 'age' in cols:
        df['Age_sq'] = df_l['age'] ** 2
        if 'hemoglobin' in cols:
            df['Age_x_Hemo'] = df_l['age'] * df_l['hemoglobin']
        if 'gtp' in cols:
            df['Age_x_GTP'] = df_l['age'] * df_l['gtp']
        if 'triglyceride' in cols:
            df['Age_x_TG'] = df_l['age'] * df_l['triglyceride']
    
    # 중성지방, 혈당
    if 'triglyceride' in cols:
        df['TG_log'] = np.log1p(df_l['triglyceride'])
    fbs_cols = [c for c in cols if 'blood' in c or 'fasting' in c]
    if len(fbs_cols) > 0:
        df['FBS_log'] = np.log1p(df_l[fbs_cols[0]])
    
    # 건강 통계
    health_cols = [c for c in cols if c in ['systolic','diastolic','hemoglobin','triglyceride','cholesterol','hdl','gtp']]
    if len(health_cols) >= 3:
        df['Health_mean'] = df_l[health_cols].mean(axis=1)
        df['Health_std'] = df_l[health_cols].std(axis=1)
    
    return df.fillna(0).replace([np.inf, -np.inf], 0)

X_fe = create_features(X)
X_test_fe = create_features(X_test)
print(f"피처 엔지니어링 후: {X_fe.shape[1]}개")

In [ ]:
# 스케일링
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_fe)
X_test_scaled = scaler.transform(X_test_fe)

print(f"Train: {X_scaled.shape}, Test: {X_test_scaled.shape}")

## 📌 STEP 4: 하이퍼파라미터 튜닝 (⭐ V3 스타일)

In [ ]:
print("=" * 50)
print("🔧 하이퍼파라미터 튜닝 (Accuracy 기준)")
print("=" * 50)

# XGBoost
print("\n🔧 XGBoost 튜닝 중...")
xgb_params = {
    'n_estimators': [300, 500, 700],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.6, 0.7, 0.8],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'gamma': [0, 0.1, 0.2],
    'reg_alpha': [0, 0.1],
    'reg_lambda': [1, 2]
}
xgb_search = RandomizedSearchCV(
    XGBClassifier(random_state=42, verbosity=0, use_label_encoder=False, eval_metric='logloss'),
    xgb_params, n_iter=60, cv=5, scoring='accuracy', random_state=42, n_jobs=-1, verbose=1
)
xgb_search.fit(X_scaled, y)
best_xgb_params = xgb_search.best_params_
print(f"XGBoost 최고: {xgb_search.best_score_:.5f}")

In [ ]:
# LightGBM
print("\n🔧 LightGBM 튜닝 중...")
lgb_params = {
    'n_estimators': [300, 500, 700],
    'max_depth': [3, 5, 7, -1],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'num_leaves': [15, 31, 63],
    'min_child_samples': [10, 20, 30],
    'subsample': [0.6, 0.7, 0.8],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'reg_alpha': [0, 0.1],
    'reg_lambda': [0, 0.1]
}
lgb_search = RandomizedSearchCV(
    LGBMClassifier(random_state=42, verbose=-1),
    lgb_params, n_iter=60, cv=5, scoring='accuracy', random_state=42, n_jobs=-1, verbose=1
)
lgb_search.fit(X_scaled, y)
best_lgb_params = lgb_search.best_params_
print(f"LightGBM 최고: {lgb_search.best_score_:.5f}")

In [ ]:
# CatBoost
print("\n🔧 CatBoost 튜닝 중...")
cat_params = {
    'n_estimators': [300, 500, 700],
    'max_depth': [4, 5, 6, 7],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'l2_leaf_reg': [1, 3, 5]
}
cat_search = RandomizedSearchCV(
    CatBoostClassifier(random_state=42, verbose=0),
    cat_params, n_iter=40, cv=5, scoring='accuracy', random_state=42, n_jobs=-1, verbose=1
)
cat_search.fit(X_scaled, y)
best_cat_params = cat_search.best_params_
print(f"CatBoost 최고: {cat_search.best_score_:.5f}")

In [ ]:
# Random Forest
print("\n🔧 Random Forest 튜닝 중...")
rf_params = {
    'n_estimators': [300, 500],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    rf_params, n_iter=30, cv=5, scoring='accuracy', random_state=42, n_jobs=-1, verbose=1
)
rf_search.fit(X_scaled, y)
best_rf_params = rf_search.best_params_
print(f"RF 최고: {rf_search.best_score_:.5f}")

In [ ]:
print("\n" + "=" * 50)
print("📊 튜닝 결과 요약")
print("=" * 50)
print(f"XGBoost:  {xgb_search.best_score_:.5f}")
print(f"LightGBM: {lgb_search.best_score_:.5f}")
print(f"CatBoost: {cat_search.best_score_:.5f}")
print(f"RF:       {rf_search.best_score_:.5f}")

## 📌 STEP 5: K-Fold 교차 예측 (⭐ 최적 파라미터 사용)

In [ ]:
print("=" * 50)
print("🎯 K-Fold 교차 예측 (튜닝된 파라미터 사용)")
print("=" * 50)

N_SPLITS = 5
SEEDS = [42, 123, 456, 789, 1004]

# OOF 예측
oof_xgb = np.zeros(len(X_scaled))
oof_lgb = np.zeros(len(X_scaled))
oof_cat = np.zeros(len(X_scaled))
oof_rf = np.zeros(len(X_scaled))

# Test 예측
test_xgb = np.zeros(len(X_test_scaled))
test_lgb = np.zeros(len(X_test_scaled))
test_cat = np.zeros(len(X_test_scaled))
test_rf = np.zeros(len(X_test_scaled))

for seed_idx, seed in enumerate(SEEDS):
    print(f"\n--- Seed {seed} ({seed_idx+1}/{len(SEEDS)}) ---")
    kfold = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    
    for fold, (tr_idx, va_idx) in enumerate(kfold.split(X_scaled, y)):
        X_tr, X_va = X_scaled[tr_idx], X_scaled[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        
        # XGBoost (튜닝된 파라미터)
        xgb_m = XGBClassifier(**best_xgb_params, random_state=seed, verbosity=0, use_label_encoder=False)
        xgb_m.fit(X_tr, y_tr)
        oof_xgb[va_idx] += xgb_m.predict_proba(X_va)[:, 1] / len(SEEDS)
        test_xgb += xgb_m.predict_proba(X_test_scaled)[:, 1] / (N_SPLITS * len(SEEDS))
        
        # LightGBM
        lgb_m = LGBMClassifier(**best_lgb_params, random_state=seed, verbose=-1)
        lgb_m.fit(X_tr, y_tr)
        oof_lgb[va_idx] += lgb_m.predict_proba(X_va)[:, 1] / len(SEEDS)
        test_lgb += lgb_m.predict_proba(X_test_scaled)[:, 1] / (N_SPLITS * len(SEEDS))
        
        # CatBoost
        cat_m = CatBoostClassifier(**best_cat_params, random_state=seed, verbose=0)
        cat_m.fit(X_tr, y_tr)
        oof_cat[va_idx] += cat_m.predict_proba(X_va)[:, 1] / len(SEEDS)
        test_cat += cat_m.predict_proba(X_test_scaled)[:, 1] / (N_SPLITS * len(SEEDS))
        
        # RF
        rf_m = RandomForestClassifier(**best_rf_params, random_state=seed, n_jobs=-1)
        rf_m.fit(X_tr, y_tr)
        oof_rf[va_idx] += rf_m.predict_proba(X_va)[:, 1] / len(SEEDS)
        test_rf += rf_m.predict_proba(X_test_scaled)[:, 1] / (N_SPLITS * len(SEEDS))

print("\n✅ K-Fold 완료!")
print("\n📊 각 모델 OOF Accuracy:")
for name, oof in [('XGBoost', oof_xgb), ('LightGBM', oof_lgb), ('CatBoost', oof_cat), ('RF', oof_rf)]:
    acc = accuracy_score(y, (oof >= 0.5).astype(int))
    print(f"{name}: {acc:.5f}")

## 📌 STEP 6: Optuna로 가중치/임계값 최적화 (⭐)

In [ ]:
print("=" * 50)
print("🎯 Optuna로 가중치 + 임계값 동시 최적화")
print("=" * 50)

def objective(trial):
    # 가중치 탐색
    w1 = trial.suggest_float('w_xgb', 0.1, 0.5)
    w2 = trial.suggest_float('w_lgb', 0.1, 0.5)
    w3 = trial.suggest_float('w_cat', 0.1, 0.5)
    w4 = 1 - w1 - w2 - w3
    
    if w4 < 0.05:  # RF 가중치 최소 5%
        return 0
    
    # 임계값 탐색
    threshold = trial.suggest_float('threshold', 0.35, 0.55)
    
    # 앙상블 예측
    oof_prob = w1*oof_xgb + w2*oof_lgb + w3*oof_cat + w4*oof_rf
    oof_pred = (oof_prob >= threshold).astype(int)
    
    return accuracy_score(y, oof_pred)

# Optuna 최적화
sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler)
study.optimize(objective, n_trials=200, show_progress_bar=True)

# 최적 결과
best_params = study.best_params
best_score = study.best_value

w1 = best_params['w_xgb']
w2 = best_params['w_lgb']
w3 = best_params['w_cat']
w4 = 1 - w1 - w2 - w3
best_threshold = best_params['threshold']

print(f"\n🏆 Optuna 최적 결과:")
print(f"   가중치: XGB={w1:.3f}, LGB={w2:.3f}, CAT={w3:.3f}, RF={w4:.3f}")
print(f"   임계값: {best_threshold:.3f}")
print(f"   OOF Accuracy: {best_score:.5f}")

## 📌 STEP 7: Stacking 앙상블

In [ ]:
print("=" * 50)
print("🎯 Stacking 앙상블")
print("=" * 50)

# Stacking 입력
oof_stack = np.column_stack([oof_xgb, oof_lgb, oof_cat, oof_rf])
test_stack = np.column_stack([test_xgb, test_lgb, test_cat, test_rf])

# Meta 모델
meta_model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
meta_model.fit(oof_stack, y)

oof_meta = meta_model.predict_proba(oof_stack)[:, 1]
test_meta = meta_model.predict_proba(test_stack)[:, 1]

print(f"Meta 모델 가중치: {meta_model.coef_[0].round(3)}")

In [ ]:
# Optuna 가중 평균 vs Stacking 비교
oof_weighted = w1*oof_xgb + w2*oof_lgb + w3*oof_cat + w4*oof_rf

print("\n📊 앙상블 방법 비교:")
for name, oof, thresh in [('Optuna 가중평균', oof_weighted, best_threshold), 
                           ('Stacking', oof_meta, best_threshold)]:
    pred = (oof >= thresh).astype(int)
    acc = accuracy_score(y, pred)
    print(f"{name}: {acc:.5f}")

# 더 좋은 방법 선택
acc_weighted = accuracy_score(y, (oof_weighted >= best_threshold).astype(int))
acc_stack = accuracy_score(y, (oof_meta >= best_threshold).astype(int))

if acc_weighted >= acc_stack:
    final_oof = oof_weighted
    final_test = w1*test_xgb + w2*test_lgb + w3*test_cat + w4*test_rf
    best_method = 'Optuna 가중평균'
else:
    final_oof = oof_meta
    final_test = test_meta
    best_method = 'Stacking'

print(f"\n✅ 선택된 방법: {best_method}")

## 📌 STEP 8: 임계값 미세 조정

In [ ]:
print("=" * 50)
print("🔍 임계값 미세 조정 (0.005 단위)")
print("=" * 50)

results = []
for thresh in np.arange(best_threshold - 0.05, best_threshold + 0.05, 0.005):
    pred = (final_oof >= thresh).astype(int)
    acc = accuracy_score(y, pred)
    f1 = f1_score(y, pred)
    results.append({'threshold': thresh, 'accuracy': acc, 'f1': f1})

results_df = pd.DataFrame(results)
best_row = results_df.loc[results_df['accuracy'].idxmax()]
final_threshold = best_row['threshold']
final_acc = best_row['accuracy']

print("\n상위 10개 임계값:")
print(results_df.nlargest(10, 'accuracy').to_string(index=False))

print(f"\n🏆 최종 임계값: {final_threshold:.3f}")
print(f"   OOF Accuracy: {final_acc:.5f}")

## 📌 STEP 9: 제출 파일 생성 (3개)

In [ ]:
print("=" * 50)
print("📝 3개 제출 파일 생성")
print("=" * 50)

thresholds = [
    round(final_threshold - 0.015, 3),
    round(final_threshold, 3),
    round(final_threshold + 0.015, 3)
]

file_paths = []

for i, thresh in enumerate(thresholds):
    pred = (final_test >= thresh).astype(int)
    oof_pred = (final_oof >= thresh).astype(int)
    oof_acc = accuracy_score(y, oof_pred)
    
    sub_df = submission.copy()
    sub_df['label'] = pred
    sub_df['label'] = sub_df['label'].astype(int)
    
    thresh_str = f"{thresh:.3f}".replace('.', '')
    filename = f'submission_v5_t{thresh_str}.csv'
    filepath = result_path + filename
    sub_df.to_csv(filepath, index=False)
    file_paths.append(filepath)
    
    marker = "⭐" if i == 1 else "  "
    n_smoking = (pred == 1).sum()
    print(f"\n{marker} 파일 {i+1}: {filename}")
    print(f"   임계값: {thresh:.3f}")
    print(f"   OOF Accuracy: {oof_acc:.5f}")
    print(f"   흡연 예측: {n_smoking}명 ({n_smoking/len(pred)*100:.1f}%)")

print("\n✅ 3개 파일 생성 완료!")

## 📌 STEP 10: 다운로드

In [ ]:
from google.colab import files

for filepath in file_paths:
    files.download(filepath)

print("\n" + "=" * 60)
print("🎉 V5 완료!")
print("=" * 60)
print(f"\n📊 최종 설정:")
print(f"   방법: {best_method}")
print(f"   가중치: XGB={w1:.3f}, LGB={w2:.3f}, CAT={w3:.3f}, RF={w4:.3f}")
print(f"   최적 임계값: {final_threshold:.3f}")
print(f"   OOF Accuracy: {final_acc:.5f}")
print(f"\n📁 생성된 파일:")
for i, (fp, th) in enumerate(zip(file_paths, thresholds)):
    marker = "👉" if i == 1 else "  "
    print(f"{marker} {fp.split('/')[-1]}")
print(f"\n🍀 행운을 빕니다!")